# 12주차: 설명의 한계와 프라이버시

- CPU 실행, Python 표준 라이브러리만 사용, GPU·설치·모델 호출 불필요
- 작성한 응답 예시와 상정 참여 레코드 사용. 실제 모델 출력·개인자료 없음
- 수정할 곳: 노이즈 multiplier 숫자 하나. 변경 없이 전체 실행 가능
- 공개 제주 문서 원문·저자 목록: https://ko.wikipedia.org/w/index.php?oldid=42464087
- 원문 CC BY-SA 4.0, 실행 시 출처와 판 번호 출력, 원문 저장 없음
- 논문: https://arxiv.org/abs/2305.04388
- DP 원리: https://www.cis.upenn.edu/~aaroth/Papers/privacybook.pdf 2.3, 3.3, 3.5절


## 실행 및 제출 기록

- 작성일 2026-11-21. 원본 16셀과 계산 흐름 유지.
- 코드 보완: 교수 개인 이메일이 들어간 User-Agent를 공개 수업 저장소 주소로 교체. 두 비교 실행에 같은 값 적용.
- 먼저 multiplier 1.0으로 전체 실행하고, 이후 TODO 한 곳을 3.0으로 바꿔 새 커널에서 전체 실행. 최종 코드 출력은 3.0 실행이며 1.0 실행 표는 결과 해석 칸에 별도 보존.
- Python 3.10.0 및 표준 라이브러리만 사용. 모델 호출, 모델 학습, 실제 개인자료 수집 없음.
- 프로젝트 표는 실제 팀 정보가 제공되지 않은 상태에서 작성한 수업용 가정. 실제 팀 회의, 멘토 확인, 사용자의 독립적인 설명 능력이나 경험을 기록한 것이 아님.
- 공개 실습 노트북을 실행하고 GitHub 과제 폴더에 제출하는 기록.
- 생성형 AI의 코드 실행 및 결과 정리 보조 사용.


## 1. 공개 문서의 출처 확인

- 행정시 이름을 공개 문서에서 확인. 응답 예시는 원문 인용이 아닌 작성 예시
- 네트워크 실패는 미확인으로 출력. 예시 실행이 성공해도 공개 사실 확인 성공을 주장하지 않음


In [1]:
import html, json, math, random, re, statistics, urllib.parse, urllib.request
OLDID = 42464087
SOURCE = f"https://ko.wikipedia.org/w/index.php?oldid={OLDID}"
print("공개 문서 및 저자 목록:", SOURCE, "라이선스: CC BY-SA 4.0")
source_verified = False
api = "https://ko.wikipedia.org/w/api.php?" + urllib.parse.urlencode({
    "action": "parse", "oldid": OLDID, "prop": "text", "format": "json", "formatversion": 2})
try:
    request = urllib.request.Request(api, headers={"User-Agent": "deepnlp-2026/1.0 (https://github.com/halla-ai/deepnlp-2026)"})
    with urllib.request.urlopen(request, timeout=30) as response:
        parsed = json.load(response)["parse"]
    assert parsed["revid"] == OLDID, "판 번호 불일치"
    source_text = html.unescape(re.sub(r"<[^>]+>", " ", parsed["text"]))
    assert "제주시" in source_text and "서귀포시" in source_text, "범주 이름 확인 실패"
    source_verified = True
    print("원문 판 번호 확인:", parsed["revid"], "범주 이름: 제주시, 서귀포시")
    del source_text, parsed
except (OSError, ValueError, KeyError, AssertionError) as error:
    print("공개 문서 조회 미확인:", type(error).__name__)
print("이후 자료: 교수 작성 응답과 상정 레코드, 실제 모델·개인 응답 아님")


공개 문서 및 저자 목록: https://ko.wikipedia.org/w/index.php?oldid=42464087 라이선스: CC BY-SA 4.0


원문 판 번호 확인: 42464087 범주 이름: 제주시, 서귀포시
이후 자료: 교수 작성 응답과 상정 레코드, 실제 모델·개인 응답 아님


## 2. 서술-결과 불일치 프록시

- 설명에서 주장한 수와 최종 답의 수만 비교하는 좁은 규칙
- reference_count=2는 교육 예제의 정답 기준, 최신 행정 변화를 추정하는 모델 아님
- 내부 계산 접근 없음. 모든 예시의 내부 충실성은 미측정


In [2]:
reference_count = 2
examples = [
    {"id": "A", "reason_text": "제주시와 서귀포시, 행정시 수는 2.", "reason_count": 2, "answer_count": 2},
    {"id": "B", "reason_text": "행정시 수는 3.", "reason_count": 3, "answer_count": 2},
    {"id": "C", "reason_text": "제주시와 서귀포시, 행정시 수는 2.", "reason_count": 2, "answer_count": 3},
    {"id": "D", "reason_text": "행정시 수는 3.", "reason_count": 3, "answer_count": 3},
]
print("작성 예시 | 설명 수 | 답 수 | 설명 수 정확 | 답 정확 | 불일치 | 내부 충실성")
for e in examples:
    e["mismatch"] = e["reason_count"] != e["answer_count"]
    print(e["id"], e["reason_count"], e["answer_count"],
          e["reason_count"] == reference_count, e["answer_count"] == reference_count,
          e["mismatch"], "미측정", sep=" | ")
print("불일치:", sum(e["mismatch"] for e in examples), "/", len(examples))
print("B: 정답이어도 설명 불일치. D: 불일치 없어도 오답. 내부 faithfulness 측정값 아님.")


작성 예시 | 설명 수 | 답 수 | 설명 수 정확 | 답 정확 | 불일치 | 내부 충실성
A | 2 | 2 | True | True | False | 미측정
B | 3 | 2 | False | True | True | 미측정
C | 2 | 3 | True | False | True | 미측정
D | 3 | 3 | False | False | False | 미측정
불일치: 2 / 4
B: 정답이어도 설명 불일치. D: 불일치 없어도 오답. 내부 faithfulness 측정값 아님.


## 3. 상정 통계와 민감도

- 공개 행정시 이름을 범주로 사용하는 작성 선호 레코드 20개
- 실제 관광 조사, 주민 의견, 학생 정보 아님
- 인접 관계: 레코드 하나 추가/제거(add/remove), 한 명의 기여 최대 1
- 공개 질의: 제주시를 선택한 count 하나. 전역 민감도 Δf=1


In [3]:
records = ["제주시"] * 12 + ["서귀포시"] * 8
true_count = sum(place == "제주시" for place in records)
sensitivity = 1.0
base_epsilon = 1.0
print("상정 레코드 수:", len(records), "제주시 count:", true_count)
for added in ("제주시", "서귀포시"):
    neighbor = records + [added]
    neighbor_count = sum(place == "제주시" for place in neighbor)
    print("하나 추가:", added, "count 변화:", abs(neighbor_count - true_count))
print("한 레코드의 0/1 기여 제한으로 민감도 1. 실제 참여 자료 없음.")


상정 레코드 수: 20 제주시 count: 12
하나 추가: 제주시 count 변화: 1
하나 추가: 서귀포시 count 변화: 0
한 레코드의 0/1 기여 제한으로 민감도 1. 실제 참여 자료 없음.


## 4. 한 숫자 변경

- 먼저 기본값 그대로 전체 실행. 다음에 1.0을 2.0 또는 3.0으로 변경하고 전체 실행
- scale = multiplier × Δf / base_epsilon
- 이상적인 메커니즘의 유효 epsilon = base_epsilon / multiplier
- 공개 seed는 재현성용. 실제 민감한 데이터의 DP 공개에는 사용할 수 없음


In [4]:
# TODO: 이 숫자만 2.0 또는 3.0으로 변경
NOISE_MULTIPLIER = 3.0


In [5]:
assert math.isfinite(NOISE_MULTIPLIER) and NOISE_MULTIPLIER > 0, "양의 유한한 숫자 사용"
scale = NOISE_MULTIPLIER * sensitivity / base_epsilon
effective_epsilon = sensitivity / scale
print("scale:", scale, "이상적 메커니즘 유효 epsilon:", effective_epsilon)
print("보장 범위 주의: 공개 seed 교육 실험, 실제 DP 공개 보장 없음")


scale: 3.0 이상적 메커니즘 유효 epsilon: 0.3333333333333333
보장 범위 주의: 공개 seed 교육 실험, 실제 DP 공개 보장 없음


## 5. 같은 표본으로 scale 변화 관찰

- 표준 Laplace 표본 z에 scale b를 곱해 noise=bz 생성
- 같은 z로 scale만 비교하므로 평균 절대 오차는 b에 정확히 비례
- 이론적 E|noise|=b, 표본 평균은 근사값
- 음수를 0으로 바꾸는 clipping은 공개 출력만 쓰는 사후 처리
- 이 노트북의 2,000회 반복은 상정 데이터 오차 관찰. 실제 개인 통계를 2,000회 공개하는 계획 아님


In [6]:
rng = random.Random(202612)
def standard_laplace():
    # 열린 구간의 교육용 inverse-CDF 표본. 운영용 안전한 DP 구현 아님.
    u = 0.0
    while u == 0.0:
        u = rng.random()
    return math.log(2 * u) if u < 0.5 else -math.log(2 * (1 - u))

standard_noise = [standard_laplace() for _ in range(2000)]
print("multiplier | scale | 유효 epsilon | 평균절대오차 | 음수출력비율 | clipping후오차")
for multiplier in dict.fromkeys([1.0, NOISE_MULTIPLIER]):
    b = multiplier * sensitivity / base_epsilon
    released = [true_count + b * z for z in standard_noise]
    mae = statistics.mean(abs(value - true_count) for value in released)
    clipped = [max(0.0, value) for value in released]
    clipped_mae = statistics.mean(abs(value - true_count) for value in clipped)
    negative_rate = sum(value < 0 for value in released) / len(released)
    print(f"{multiplier:.1f} | {b:.1f} | {sensitivity / b:.3f} | {mae:.3f} | {negative_rate:.3f} | {clipped_mae:.3f}")
print("표준표본 평균 절댓값:", round(statistics.mean(abs(z) for z in standard_noise), 6))
print("같은 표본 사용: 변경 전후 clipping 전 MAE 비율 = multiplier 비율")


multiplier | scale | 유효 epsilon | 평균절대오차 | 음수출력비율 | clipping후오차
1.0 | 1.0 | 1.000 | 0.969 | 0.000 | 0.969
3.0 | 3.0 | 0.333 | 2.906 | 0.006 | 2.891
표준표본 평균 절댓값: 0.96873
같은 표본 사용: 변경 전후 clipping 전 MAE 비율 = multiplier 비율


## 6. 반복 공개와 판단 보류

- 이상적인 epsilon-DP 독립 공개 k회의 기본 순차 합성 상한: k × epsilon
- 여러 응답의 평균을 공개한다고 예산이 사라지지 않음
- 공개 seed로 계산한 아래 숫자는 예산 산술 관찰이며 보호된 서비스가 아님
- 실제 프로젝트는 노이즈를 더하기 전에 수집하지 않을 자료와 사람 확인 경계를 먼저 설정


In [7]:
for k in (1, 5, 10):
    print("공개 횟수:", k, "기본 순차 합성 epsilon 상한:", round(k * effective_epsilon, 3))
print("허용 예시: 공개 문서 출처와 확인 시점 안내")
print("보류 예시: 안전·자격·권리 최종 판정, 지역 정책 수용성 자동 확정")
print("출처 확인 상태:", "확인" if source_verified else "미확인")
print("완료: 내부 충실성 미측정, 실제 개인정보 없음, 실제 DP 서비스 보장 없음")


공개 횟수: 1 기본 순차 합성 epsilon 상한: 0.333
공개 횟수: 5 기본 순차 합성 epsilon 상한: 1.667
공개 횟수: 10 기본 순차 합성 epsilon 상한: 3.333
허용 예시: 공개 문서 출처와 확인 시점 안내
보류 예시: 안전·자격·권리 최종 판정, 지역 정책 수용성 자동 확정
출처 확인 상태: 확인
완료: 내부 충실성 미측정, 실제 개인정보 없음, 실제 DP 서비스 보장 없음


## 결과 해석 기록

□ 실제 실행과 공개 원문 확인 범위

| 실행 | 코드 실행 시각 | TODO 값 | 코드 셀 | 오류 출력 | 고정 판 확인 |
|---|---|---:|---:|---:|---|
| 기준 | 2026-11-21T23:06:01+09:00 ~ 2026-11-21T23:06:31+09:00 | 1.0 | 7 | 0 | 미확인 |
| 변경 | 2026-11-21T23:06:33+09:00 ~ 2026-11-21T23:06:35+09:00 | 3.0 | 7 | 0 | 확인 |

- 두 실행은 각각 새 커널에서 1~7번 실행 번호로 수행. 난수 seed 202612, 작성 레코드 20개, count 12, add/remove 인접 관계, 한 레코드 기여 최대 1, 기준 epsilon 1 및 표본 2,000개 유지.
- 비교 실행 사이의 코드 변경은 `NOISE_MULTIPLIER = 1.0`에서 `3.0`으로 바꾼 한 곳뿐. 나머지 코드 동일.
- 공개 원문 확인은 API의 판 번호 42464087과 `제주시`, `서귀포시` 문자열 존재 확인. 최신 행정 상태, 모든 행정 구분의 완전성, 법률 적용이나 정책 정당성을 검증한 결과가 아님.
- 이번 기준 실행은 원문 요청 timeout으로 미확인, 변경 실행은 고정 판 확인에 성공. 두 실행의 원문 확인 상태를 표와 출력에 그대로 보존.
- 원문은 일시적으로 읽고 삭제한 변수로 확인, 원문 파일 저장 없음. 출처와 저자 목록 링크 및 강의에 제시된 CC BY-SA 4.0 표기는 상단과 실행 출력에 유지.

□ 변경 전후 실제 출력 표

| 실행 | multiplier | scale b | 유효 epsilon | 관찰 평균절대오차 | 음수 출력 비율 | clipping 후 오차 |
|---|---:|---:|---:|---:|---:|---:|
| 기준 전체 실행 | 1.0 | 1.0 | 1.000 | 0.969 | 0.000 | 0.969 |
| 변경 전체 실행 | 3.0 | 3.0 | 0.333 | 2.906 | 0.006 | 2.891 |

- 표는 두 실행의 출력값을 그대로 옮긴 값. 세 자리로 반올림된 표기이며 `0.000`을 모든 가능한 실행에서 음수 출력이 불가능하다는 뜻으로 읽지 않음.
- 변경 실행의 원본 출력에는 같은 표본으로 다시 계산한 1.0 행도 있고, 기준 실행의 1.0 행과 표시값 일치.
- 공식에 따른 방향 해석: 민감도와 기준 epsilon이 고정일 때 multiplier 3은 scale을 1에서 3으로 확대하고 이상적인 메커니즘의 유효 epsilon을 1에서 1/3으로 감소. 이 문장은 실행 후 정리이며, 사용자의 독립적인 실행 전 예측 답안을 관찰한 기록이 아님.
- 같은 z에서 clipping 전 오차는 `abs(b*z)`이므로 scale 3배에 따라 반올림 전 MAE도 부동소수점 반올림 범위에서 3배. 표시된 2.906/0.969의 비가 정확히 3이 아닌 것은 출력 반올림 때문.
- 이론적인 E(abs(noise))는 각각 1과 3, 이번 표본의 관찰 MAE는 0.969와 2.906. 이론 기댓값과 유한 표본 평균의 구분.
- 변경 설정의 음수 출력 비율 표시는 0.006, clipping 후 MAE는 2.891. 음수 count를 0으로 옮긴 사후 처리가 일부 오차를 줄인 관찰. clipping 자체가 이 교육용 공개 seed 실험에 실제 DP 보장을 부여하지는 않음.
- 반복 공개의 기본 순차 합성 상한: 기준 epsilon 1에서 1/5/10회는 1.000/5.000/10.000, 변경 epsilon 1/3에서 0.333/1.667/3.333. 출력에 따른 이론적 예산 산술이며, 실제 개인 통계를 공개한 기록이 아님.

기준 전체 실행의 보존 출력:

```text
scale: 1.0 이상적 메커니즘 유효 epsilon: 1.0
보장 범위 주의: 공개 seed 교육 실험, 실제 DP 공개 보장 없음
multiplier | scale | 유효 epsilon | 평균절대오차 | 음수출력비율 | clipping후오차
1.0 | 1.0 | 1.000 | 0.969 | 0.000 | 0.969
표준표본 평균 절댓값: 0.96873
같은 표본 사용: 변경 전후 clipping 전 MAE 비율 = multiplier 비율
```

□ 운영 전 바꿀 조건

1. 공개 seed 제거만으로 운영 승인을 하지 않고, 예측 가능한 교육용 `random.Random` 대신 안전한 난수와 수치 구현을 검토한 DP 구현 필요. 이번 과제에서 구현하거나 보호 성능을 실증한 항목은 아님.
2. 실제 자료를 사용한다면 한 사람이 여러 레코드를 남길 수 있는지 확인하고 기여 제한과 인접 관계를 검증해야 함. 최대 기여 1을 보장하지 못하면 민감도 1을 그대로 쓸 수 없으므로 공개 보류.
3. 공개할 count, 횟수, 누적 예산과 접근 권한을 공개 전에 정하는 설계 필요. 2,000회 반복은 작성 자료의 오차 관찰이며 2,000회 실제 개인 통계를 배포할 계획이 아님.

□ 수업용 가정 프로젝트의 위험 체크리스트

- 가정 프로젝트: 공개 제주 행정정보 문서의 출처를 찾는 QA 안내 도우미.
- 실제 팀 목적이나 기업 자료가 제공되지 않았으므로 아래 내용은 설계 초안. 검토 역할을 가정한 것이며 실제 담당자 지정, 멘토 승인 또는 운영 검증은 없음.
- 법률 항목은 강의에서 제시한 검토 질문에 대한 미확인 상태 기록. 최신 법률의 적용, 고위험 여부 또는 적합성의 확정 판단 아님.

| 검토 항목 | 상정 프로젝트 기입 | 확인 상태 |
|---|---|---|
| 사용 목적과 사용자 | 공개 행정정보의 위치와 근거를 찾는 가상 이용자에게 문서 출처와 범주 이름 안내 | 수업용 목적 가정, 실제 사용자 조사 없음 |
| 입력 자료와 기준 시점 | 제주 공개 문서 고정 판 42464087, 2026-11-21 변경 실행에서 조회 확인. 작성 응답 4개와 상정 선호 레코드 20개는 별도 표시 | 고정 판/이름 존재 확인, 최신성 미확인 |
| 결과를 사용하는 결정 | 자료 탐색 지원. 안전, 입학, 지원 자격, 권리의 최종 판정으로 사용하지 않는 범위 | 실제 후속 영향 평가 없음 |
| 적용 관할/EU 관련성 | 운영 지역, 시장 및 출력 이용의 EU 관련성에 관한 자료가 없으므로 적용 관할 검토 보류 | 미확인 |
| 금지 관행 가능성 | 개인 조작, 차별적 점수화, 감시 기능은 가정 범위에서 제외. 강의의 Article 5 검토 질문에 대한 실제 적용 검토는 미실시 | 적합성 확정 안 함 |
| Annex I/III 용도 해당성 | 현재 가정은 공개 문서 탐색. 안전 제품 구성요소, 입학/학습 성과 평가 등으로 목적이 바뀌면 새 검토 필요 | 법률 해당성 미확인 |
| 결정에 실질적 영향 | 공식 권리 판정을 대체하지 않도록 출처 안내로 출력 제한. 실제 이용자가 어떤 결정에 쓰는지는 별도 확인 필요 | 설계 제한만 제시, 실증 없음 |
| 자연인 프로파일링 | 개인 성향 추정, 순위, 점수화 기능을 설계하지 않음 | 가정상 없음, 운영 시스템 감사는 없음 |
| Article 6(3) 예외 주장 | 관할, 영향 및 예외 조건 검토가 완료되지 않아 예외 주장 보류 | 미판정 |
| 사람 검토/중단 권한 | 가상 담당 검토자가 원문과 판 번호를 확인하고 초안 수정, 응답 거부, 공개 중단을 할 수 있도록 설계 | 역할 배정과 권한 작동 미검증 |
| 투명성/이의 제기 | 작성 예시 또는 안내 초안이라는 표시, 출처 확인 상태 제공. 담당 검토자에게 정정 요청하는 창구를 운영 전 정할 계획 | 실제 창구 미구현 |
| 수집 최소화/보관 | 개인 이름, 연락처, 개별 민원 사연을 입력으로 요구하지 않는 설계. 실습 원문은 메모리 확인 후 삭제, 원문 파일 저장 없음. 운영 로그의 보관 기간과 접근 권한은 미정이므로 수집 기능 보류 | 실제 개인자료 없음, 운영 보관 정책 미확정 |
| 잠정 결론/미확인 | 수업용 초기 위험 검토 기록. 실제 운영 적합성, 고위험 여부 및 예외를 확정하지 않으며 사용 목적 변경 시 재검토 | 운영 판단 보류 |

□ 설계도 §선: 상정 공개 행정정보 QA의 경계

| 항목 | 상정 프로젝트 문장 |
|---|---|
| 허용 출력 | 시스템은 지정된 공개 문서에서 확인한 행정정보와 문서 위치를 정리하고 출처 URL, 판 번호, 조회 시점 및 확인 상태를 함께 제공. 작성 예시는 원문 인용이나 실제 모델 출력으로 표시하지 않음 |
| 넘기지 않을 판단 | 안전, 입학/지원 자격, 권리의 최종 판단과 지역 정책의 수용성/주민 동의를 자동 확정하지 않음. 공개 문서 요약을 실제 주민 의견으로 대신하지 않음 |
| 보류 조건 | 출처 확인 실패, 근거 충돌, 최신 정보가 필요한데 고정 판만 확보한 경우에는 확인 범위를 밝히고 해당 판단을 보류 |
| 확인 주체와 권한 | 가상 담당 검토자가 원자료와 시점을 확인한 뒤 수정, 거부, 공개 중단 권한을 행사하도록 설계. 실제 사람 검토나 권한 테스트를 완료했다는 주장은 없음 |
| 이의 제기 | 이용자가 공개 문서 URL과 오류 부분으로 정정/재검토를 요청하는 절차를 마련할 계획. 실제 접수 창구는 미구현이며 개인별 자격 심사 자료는 받지 않음 |
| 데이터 경계 | 개인정보와 실제 주민/학생 선호 응답 수집 없음. 현재 집계는 작성 레코드만 사용. 실데이터 집계가 필요해지면 수집 필요성, 기여 제한, 공개 예산 및 보관 조건을 먼저 검토하고 미정 상태에서는 수집/공개 보류 |
| 재검토 계기 | 사용 목적 또는 결정 영향의 변경, 새 공개 판, 근거 오류, 운영 관할/관련 법령의 변경, 개인정보 입력 기능 추가 시 위험 검토 갱신 |

- 다른 팀의 검토나 멘토 확인은 수행하지 않았으며, 가정된 사람 확인 절차의 작동도 미검증.
- 체크리스트를 작성한 사실과 법률 적합성, 실제 개인정보 보호, 주민 동의 또는 시스템 안전을 확인한 사실의 구분.


## 돌아보기: 확인 질문과 답변

1. **B와 D의 설명-결과 불일치와 최종 답 정확성은 어떻게 다른가?**

   B는 설명 수 3과 최종 답 수 2가 달라 불일치 프록시가 True. 최종 답 2는 교육용 reference_count=2와 같아 답 정확성은 True지만, 설명의 숫자는 그 기준에 대해 틀림.

   D는 설명 수 3과 최종 답 수 3이 같아 불일치 프록시가 False. 그러나 최종 답 3은 기준 2와 달라 답 정확성은 False이고 설명의 숫자도 틀림. 따라서 이 프록시 통과는 정답 보장이 아님.

   네 행은 교수 작성 응답 예시이며 실제 모델이 생성한 표본이 아님. 코드가 비교하는 것은 수동으로 제공된 reason_count와 answer_count이고, reason_text 전체를 분석하거나 모델 내부 계산에 접근하지 않음. A~D 모두 내부 faithfulness는 미측정이며, 불일치 2/4를 모델의 환각률이나 내부 충실성 점수로 부를 수 없음.

2. **multiplier 1과 3의 scale, 유효 epsilon, 관찰 MAE는?**

   민감도 1과 기준 epsilon 1에서 multiplier 1은 scale 1, 유효 epsilon 1, 관찰 MAE 0.969. multiplier 3은 scale 3, 유효 epsilon 1/3(표시 0.333), 관찰 MAE 2.906. 고정 seed로 뽑은 표준 Laplace 표본 2,000개가 같아 clipping 전 MAE는 scale 비율대로 약 3배가 됨.

   이상적인 연속 Laplace 메커니즘에서는 작은 유효 epsilon이 더 엄격한 프라이버시 제약에 대응하지만, 이 공개 seed 교육 계산이 실제 DP 보호를 제공했다는 뜻은 아님. 이론 E(abs(noise))는 설정별 1과 3이며, 관찰한 0.969와 2.906은 유한 표본 값. clipping 후 변경 오차 2.891은 clipping 전 2.906과 별도로 보고.

3. **공개 seed의 재현 가능한 실험이 실제 DP 공개 보장이 아닌 이유는?**

   seed 202612와 코드, scale을 아는 관찰자는 같은 난수를 재현할 수 있으므로 실제 민감한 count에 더한 노이즈를 예측하거나 제거할 수 있음. 이번 레코드 20개는 작성한 자료이므로 실제 개인의 보호 여부를 검증할 대상도 없음.

   seed를 숨기는 것만으로 운영 조건이 갖춰지는 것도 아님. 안전한 난수와 수치 구현, 인접 관계에 맞는 실제 기여 제한, 공개 횟수와 누적 예산, 접근 통제를 따로 확인해야 함. 1/5/10회 합성 상한은 이상적인 메커니즘의 예산 산술이며, 표본 2,000개를 만든 것이 실제 통계 2,000회 공개나 개인정보 보호 인증을 뜻하지 않음.

4. **프로젝트 체크리스트의 사용 목적, 사람 확인, 미확인 근거는?**

   사용 목적은 가정한 이용자가 공개 제주 행정정보의 출처를 찾는 QA 안내. 확인된 고정 판의 범주 이름과 출처 제공까지 허용하고 안전, 자격, 권리 및 정책 수용성의 최종 판단은 자동화하지 않는 범위.

   사람 확인은 가상 담당 검토자가 원문, 시점, 근거 충돌을 확인하고 수정/거부/중단하는 설계 조건. 실제 팀원이나 멘토가 검토했다는 기록은 없고, 사람에게 중단 권한이 실제로 작동하는지 시험한 것도 아님.

   이번 실측에서 확인한 근거는 고정 판 42464087과 두 범주 이름의 존재뿐. 최신 행정 현황, 실제 주민 의견, 정책 정당성, 운영 관할과 법률 적용, 실제 개인정보 보호 및 실제 검토 권한은 미확인 또는 미실시. 이런 상태를 실패 점수나 확인 완료로 바꾸지 않고 체크리스트와 §선에 남김.

## 제출 안내

- 코드 구현이나 자료 수집을 새로 요구하는 과제가 아닌 제공된 실습의 관찰 기록
- 실행한 노트북의 출력 표와 질문별 관찰·판정 이유를 함께 제출
- 저장된 결과에서 미확인·미판정은 그대로 표시, 모델 학습 성과로 보고하지 않는 원칙
- 학생 폴더에는 학번만 사용, 이름·연락처·데이터 원본·API 키 제외
